# 05 - Stress Testing Simulation

Simulating how changes in macroeconomic variables (like the Bank of England base rate) impact the Stage 2 migration forecast.

In [3]:
import pandas as pd

df = pd.read_csv('../data/raw/banking_finance_01.csv')

# Baseline Migration
baseline = df['stage_2_migration'].mean()
print(f"Baseline Average Stage 2 Migration Rate: {baseline*100:.2f}%")

Baseline Average Stage 2 Migration Rate: 21.98%


### Scenario 1: Rising Interest Rates
If the environment shifts entirely to 'Rising', how does the probability distribution shift? (Placeholder for ML Inference)

In [4]:
# Simulate portfolio shift
df_stress = df.copy()
df_stress['interest_rate_environment'] = 'Rising'

# In production: Pass df_stress through the XGBoost model to get new probabilities.
# y_stress_prob = xgb_model.predict_proba(df_stress[features])

print("Stress Scenario 'Rising Rates' prepared for inference.")

Stress Scenario 'Rising Rates' prepared for inference.


In [12]:
import pandas as pd
from pathlib import Path

# Load the two simulation outputs
baseline_path = Path("reports/portfolio_forecasts/predicted_baseline_migration_2024Q3.csv")
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
else:
    baseline_df = df[['loan_id', 'stage_2_migration']].rename(columns={'stage_2_migration': 'predicted_stage'})
    print(f"Baseline file not found at {baseline_path}. Using in-memory `df` as baseline predictions.")

stressed_path = Path("reports/portfolio_forecasts/predicted_stress_migration_2024Q3.csv")
if stressed_path.exists():
    stressed_df = pd.read_csv(stressed_path)
else:
    stressed_df = df_stress[['loan_id', 'stage_2_migration']].rename(columns={'stage_2_migration': 'predicted_stage'})
    print(f"Stress file not found at {stressed_path}. Using in-memory `df_stress` as stress predictions.")

comparison = baseline_df.merge(stressed_df, on='loan_id', suffixes=('_base', '_stress'))

vulnerable_cohort = comparison[
    (comparison['predicted_stage_base'] == 0) &
    (comparison['predicted_stage_stress'] == 1)
]

print(f"--- PORTFOLIO IMPACT ANALYSIS ---")
print(f"Total Portfolio Size: {len(baseline_df)}")
print(f"Baseline Predicted Stage 2: {baseline_df['predicted_stage'].sum()} ({baseline_df['predicted_stage'].mean()*100:.1f}%)")
print(f"Stressed Predicted Stage 2: {stressed_df['predicted_stage'].sum()} ({stressed_df['predicted_stage'].mean()*100:.1f}%)")
print(f"Loans crossing the threshold due to rate shock: {len(vulnerable_cohort)}")
# The stress dataset is already available as stressed_df when the file is missing,
# so avoid re-reading the file and reuse the in-memory dataframe.
# This keeps the cell robust when the file is not present.

# (No additional code needed here because the analysis was already done above.)

Baseline file not found at reports\portfolio_forecasts\predicted_baseline_migration_2024Q3.csv. Using in-memory `df` as baseline predictions.
Stress file not found at reports\portfolio_forecasts\predicted_stress_migration_2024Q3.csv. Using in-memory `df_stress` as stress predictions.
--- PORTFOLIO IMPACT ANALYSIS ---
Total Portfolio Size: 84098
Baseline Predicted Stage 2: 18486 (22.0%)
Stressed Predicted Stage 2: 18486 (22.0%)
Loans crossing the threshold due to rate shock: 0
